In [ ]:
cd ../../../../hybrid_diversity/5_snp_analyses/2_atype_network/

In [ ]:
ml StdEnv
ml vcftools/0.1.16
ml bcftools/1.20.0 

In [ ]:
# # baseline filtering
# vcftools --vcf atype_variants.vcf --remove-indels --min-alleles 2 --max-alleles 2 \
#   --recode --out step1_biallelic_snps

In [ ]:
# # quality filtering
# vcftools --vcf step1_biallelic_snps.recode.vcf --minQ 100 \
#   --recode --out step2_minQ

In [ ]:
# # add depth filtering
# vcftools --vcf step2_minQ.recode.vcf --min-meanDP 20 --minDP 10 \
#   --recode --out step3_depth

In [ ]:
# for f in step*.recode.vcf; do
#   echo -n "$f: "
#   grep -v "^#" $f | wc -l
# done

In [ ]:
# # check per-sample missingness before filtering
# vcftools --vcf step3_depth.recode.vcf --missing-indv --out missingness
# cat missingness.imiss

**Unsurprisingly, A2 and A3 strains are driving the majority of that 332k SNPs removed using maxmissing=1**

This is due to reference bias, so I will keep the strict filtering in place

In [ ]:
# combined final quality filtering
vcftools --vcf atype_variants.vcf \
  --remove-indels --min-alleles 2 --max-alleles 2 \
  --minQ 100 \
  --min-meanDP 20 \
  --max-missing 1 \
  --recode --out atype_variants_qualfiltered

In [ ]:
# quality + MAF and HW filtering
vcftools --vcf atype_variants_qualfiltered.recode.vcf \
  --maf 0.05 \
  --hwe 0.000001 \
  --recode --out atype_variants_qualfiltered_maffiltered

In [ ]:
for f in atype_variants*.recode.vcf; do
  echo -n "$f: "
  grep -v "^#" $f | wc -l
done

In [ ]:
bcftools reheader -s strain_ids.txt -o atype_variants_qualfiltered.recode.renamed.vcf atype_variants_qualfiltered.recode.vcf

In [ ]:
bcftools reheader -s strain_ids.txt -o atype_variants_qualfiltered_maffiltered.recode.renamed.vcf atype_variants_qualfiltered_maffiltered.recode.vcf

In [ ]:
bcftools reheader -s strain_ids.txt -o atype_variants.renamed.vcf atype_variants.vcf

In [ ]:
# filter vcf to only include hybrids
vcftools --vcf atype_variants_qualfiltered_maffiltered.recode.renamed.vcf \
  --remove-indv CCMP2941 \
  --remove-indv NIES1812 \
  --remove-indv RCC3703 \
  --remove-indv CC2192B4 \
  --remove-indv CCMP3037 \
  --remove-indv PK2298B4 \
  --remove-indv CC2192B5 \
  --remove-indv CC2192C1 \
  --remove-indv CC2192D3 \
  --remove-indv CC2192D2 \
  --remove-indv 12B1 \
  --recode --out atype_variants_qualfiltered_maffiltered_renamed_justhybrids

In [ ]:
bcftools stats atype_variants_qualfiltered_maffiltered.recode.renamed.vcf | grep "^SN"

In [ ]:
bcftools stats atype_variants_qualfiltered_maffiltered_renamed_justhybrids.recode.vcf | grep "^SN"

In [ ]:
bcftools query -l atype_variants_qualfiltered_maffiltered_renamed_justhybrids.recode.vcf